In [0]:
# Alpha Vantage Parameters

# ❚ Required: function
# The function of your choice. In this case, function=TREASURY_YIELD

# ❚ Optional: interval
# By default, interval=monthly. Strings daily, weekly, and monthly are accepted.

# ❚ Optional: maturity
# By default, maturity=10year. Strings 3month, 2year, 5year, 7year, 10year, and 30year are accepted.

# ❚ Optional: datatype
# By default, datatype=json. Strings json and csv are accepted with the following specifications:
# json returns the time series in JSON format;
# csv returns the time series as a CSV (comma separated value) file.

In [0]:
import requests


# API key for Alpha Vantage
api_key = ""

# Define the base URL and parameters
function = "TREASURY_YIELD"
interval = "daily"
maturity = "5year"
datatype = "csv"

In [0]:
# Construct the URL
url = f"https://www.alphavantage.co/query?function={function}&interval={interval}&maturity={maturity}&datatype={datatype}&apikey={api_key}"

# Make the API request
r = requests.get(url)
data = r.text

# Print the data
print("\n".join(data.splitlines()[:5]))

In [0]:
import pandas as pd
import io
from pyspark.sql import functions as F

# Read CSV
df = pd.read_csv(io.StringIO(data))

# Convert 'timestamp' column to datetime
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce").dt.date

# Convert 'value' column to float, coerce errors to NaN
df["value"] = pd.to_numeric(df["value"], errors="coerce")

# Pandas → Spark
spark_df = spark.createDataFrame(df)

# Add audit columns in Spark
spark_df = spark_df.withColumn(
    "CreatedAt",
    F.date_trunc(
        "second", F.from_utc_timestamp(F.current_timestamp(), "Europe/Berlin")
    ),
).withColumn("UpdatedAt", F.lit(None).cast("timestamp"))
table_name = "thekitchen.av.five_year_treasury"

try:
    (
        spark_df.write.format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(table_name)
    )
    print(f"Table '{table_name}' written successfully.")
except Exception as e:
    print(f"Failed to write table '{table_name}': {e}")

In [0]:
%skip
table_name = "av.five_year_treasury"

spark.sql(f"DROP TABLE IF EXISTS {table_name}")

In [0]:
%skip
display(spark_df.orderBy("timestamp", ascending=False).limit(10))